In [1]:
# =============================================================================
# CSetup & Multi-Document Structured Extraction
# =============================================================================
# Real claims don't arrive as clean single documents. You get a police report,
# a medical record, a witness statement, and an adjuster's notes — all about
# the same incident. This cell extracts structured data across multiple sources.
# =============================================================================

import boto3
import json
import re
from datetime import datetime

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
model_id = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

def ask(prompt, system=None, max_tokens=2048, temperature=0.0):
    kwargs = {
        'modelId': model_id,
        'messages': [{'role': 'user', 'content': [{'text': prompt}]}],
        'inferenceConfig': {'maxTokens': max_tokens, 'temperature': temperature}
    }
    if system:
        kwargs['system'] = [{'text': system}]
    response = bedrock.converse(**kwargs)
    return {
        'text': response['output']['message']['content'][0]['text'],
        'input_tokens': response['usage']['inputTokens'],
        'output_tokens': response['usage']['outputTokens'],
        'stop_reason': response['stopReason']
    }

def parse_llm_json(text):
    """Strip markdown fencing and parse JSON from LLM output."""
    raw = text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)

# --- Multiple source documents for the same incident ---

police_report = """
PORTLAND POLICE BUREAU - TRAFFIC CRASH REPORT
Report #: PPB-2026-008834
Date/Time: 02/22/2026, 14:35 hours
Location: NW 23rd Ave & NW Lovejoy St, Portland OR 97210
Weather: Overcast, dry pavement

VEHICLE 1: 2024 Toyota Camry, Oregon plate ABC-1234
Driver: Patricia Yamamoto, DOB 06/15/1978, DL# OR-889-2234
Insurance: Evergreen Mutual, Policy EM-AUTO-2025-44891

VEHICLE 2: 2021 Chevrolet Silverado, Oregon plate XYZ-9876  
Driver: Marcus Webb, DOB 11/03/1995, DL# OR-445-7891
Insurance: Pacific Shield Insurance, Policy PS-2024-10283

DESCRIPTION: V1 was traveling southbound on NW 23rd approaching Lovejoy. 
V2 was exiting a parallel parking space on the east side of NW 23rd and 
pulled into the travel lane. V1 swerved left to avoid but V2 struck V1 
on the right front quarter panel. V1 then struck a parked vehicle (V3, 
unoccupied 2019 Honda CR-V, plate DEF-5555, owner: Linda Chow). 

V1 driver complained of neck pain at scene. V2 driver stated he did not 
see V1 approaching. No passengers in either vehicle.

CITATIONS: V2 driver cited for Failure to Yield When Entering Roadway 
(ORS 811.550). No citations for V1 driver.

RESPONDING OFFICER: Ofc. Daniel Reeves, Badge #4472
"""

medical_record = """
PROVIDENCE PORTLAND MEDICAL CENTER
EMERGENCY DEPARTMENT VISIT

Patient: Patricia Yamamoto, DOB: 06/15/1978
Date of Visit: 02/22/2026, 16:10
MRN: PPM-2026-118834
Referring: Self-presented, motor vehicle accident

CHIEF COMPLAINT: Neck pain and headache following MVA approximately 90 
minutes prior.

EXAMINATION: 
- Cervical spine: Tenderness at C4-C6, limited ROM, no step-off deformity
- Neurological: Alert, oriented x4, cranial nerves intact, grip strength 
  5/5 bilateral, no numbness or tingling
- Imaging: C-spine X-ray shows no fracture or dislocation. Mild loss of 
  normal lordosis suggesting muscle spasm.

DIAGNOSIS:
1. Cervical strain/sprain (whiplash), ICD-10: S13.4XXA
2. Tension headache secondary to trauma, ICD-10: G44.209

TREATMENT: Cervical collar provided. Ibuprofen 600mg TID prescribed.
REFERRAL: Physical therapy, 2x weekly for 6 weeks.
FOLLOW-UP: PCP in 1 week if symptoms persist. Return to ED if 
numbness, weakness, or severe headache develops.

DISCHARGE: 02/22/2026, 18:45
ATTENDING: Dr. Sarah Okonkwo, MD
BILLING: $2,840 (ER visit, imaging, supplies)
"""

adjuster_notes = """
CLAIM FILE NOTES - Evergreen Mutual
Claim #: EM-CLM-2026-02891
Adjuster: Rachel Torres, License #ADJ-OR-11283
Date: 02/24/2026

INITIAL CONTACT: Spoke with insured Patricia Yamamoto at 10:15 AM. She is 
coherent and cooperative. She confirmed the police report account — she was 
driving normally when the truck pulled out in front of her. She estimates 
her speed at 25 mph (posted limit is 25). She is wearing the cervical collar 
and has her first PT appointment scheduled for 02/28.

VEHICLE INSPECTION: Scheduled for 02/26 at Eastside Auto Body. Photos from 
insured show significant right front quarter panel damage, cracked headlight 
assembly, bent fender. Airbags did not deploy. Vehicle is drivable.

PRELIMINARY ESTIMATE: $6,800-$8,200 for V1 repairs based on photos.

THIRD VEHICLE: Contacted Linda Chow regarding V3 damage. Rear bumper 
cracked, taillight broken. She is filing through her own carrier (Cascade 
Insurance) and they will subrogate against our claim. Estimated V3 damage: 
$2,100-$2,800.

LIABILITY ASSESSMENT: Clear liability on V2 driver (Webb). Police citation 
supports. Will pursue subrogation against Pacific Shield for all costs.

RED FLAGS: None identified. Consistent statements, reasonable treatment, 
prompt reporting (filed claim same day as accident).

RESERVES SET:
- Property damage (V1): $8,500
- Property damage (V3 subrogation exposure): $3,000
- Medical/BI: $12,000
- Total initial reserve: $23,500
"""

# --- Multi-document extraction ---
print("=" * 70)
print("MULTI-DOCUMENT STRUCTURED EXTRACTION")
print("=" * 70)

extraction_prompt = f"""You have three documents about the same auto insurance claim.
Cross-reference them and extract a unified claim record as JSON.
Return ONLY valid JSON, no markdown fencing.

Where documents provide conflicting or complementary information, use the 
most authoritative source (police report > medical record > adjuster notes).
Flag any discrepancies in a "discrepancies" array.

Schema:
{{
  "claim": {{
    "claim_number": "string",
    "incident_date": "YYYY-MM-DD",
    "incident_time": "HH:MM",
    "location": "string",
    "police_report_number": "string"
  }},
  "insured": {{
    "name": "string",
    "dob": "YYYY-MM-DD",
    "policy_number": "string",
    "carrier": "string",
    "vehicle": {{"year": number, "make": "string", "model": "string", "plate": "string"}}
  }},
  "other_party": {{
    "name": "string",
    "dob": "YYYY-MM-DD",
    "policy_number": "string",
    "carrier": "string",
    "vehicle": {{"year": number, "make": "string", "model": "string", "plate": "string"}},
    "citations": ["string"]
  }},
  "third_party_property": {{
    "owner": "string",
    "vehicle": {{"year": number, "make": "string", "model": "string", "plate": "string"}},
    "carrier": "string",
    "estimated_damage": {{"low": number, "high": number}}
  }},
  "injuries": [
    {{
      "person": "string",
      "diagnoses": [{{"description": "string", "icd10": "string"}}],
      "treatment": "string",
      "provider": "string",
      "medical_cost": number
    }}
  ],
  "liability": {{
    "assessment": "string",
    "our_insured_pct": number,
    "other_party_pct": number,
    "basis": "string"
  }},
  "financials": {{
    "vehicle_repair_estimate": {{"low": number, "high": number}},
    "medical_costs_to_date": number,
    "projected_medical": "string",
    "reserves": {{
      "property_damage": number,
      "third_party_property": number,
      "medical_bi": number,
      "total": number
    }}
  }},
  "flags": {{
    "red_flags": ["string"],
    "subrogation_potential": boolean,
    "subrogation_target": "string"
  }},
  "discrepancies": ["string"],
  "source_documents": ["string"],
  "extraction_confidence": "HIGH | MEDIUM | LOW"
}}

DOCUMENT 1 — POLICE REPORT:
{police_report}

DOCUMENT 2 — MEDICAL RECORD:
{medical_record}

DOCUMENT 3 — ADJUSTER NOTES:
{adjuster_notes}"""

result = ask(extraction_prompt, max_tokens=3000)
print(result['text'][:200] + "..." if len(result['text']) > 200 else result['text'])
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# Parse and display key fields
try:
    claim = parse_llm_json(result['text'])
    print("\n✓ Valid JSON — Unified claim record extracted")
    print(f"\n  CLAIM: {claim['claim']['claim_number']}")
    print(f"  INSURED: {claim['insured']['name']}")
    print(f"  OTHER PARTY: {claim['other_party']['name']}")
    print(f"  LIABILITY: {claim['liability']['our_insured_pct']}% ours / {claim['liability']['other_party_pct']}% theirs")
    print(f"  TOTAL RESERVE: ${claim['financials']['reserves']['total']:,.2f}")
    print(f"  SUBROGATION: {'Yes → ' + claim['flags']['subrogation_target'] if claim['flags']['subrogation_potential'] else 'No'}")
    print(f"  CONFIDENCE: {claim['extraction_confidence']}")
    
    if claim.get('discrepancies'):
        print(f"\n  DISCREPANCIES FOUND ({len(claim['discrepancies'])}):")
        for d in claim['discrepancies']:
            print(f"    ⚠ {d}")
    else:
        print(f"\n  NO DISCREPANCIES — sources are consistent")

    print(f"\n  DIAGNOSES:")
    for injury in claim['injuries']:
        for dx in injury['diagnoses']:
            print(f"    • {dx['description']} ({dx['icd10']})")

except json.JSONDecodeError as e:
    print(f"\n✗ JSON parse error: {e}")

MULTI-DOCUMENT STRUCTURED EXTRACTION
{
  "claim": {
    "claim_number": "EM-CLM-2026-02891",
    "incident_date": "2026-02-22",
    "incident_time": "14:35",
    "location": "NW 23rd Ave & NW Lovejoy St, Portland OR 97210",
    "police_r...

[Tokens: 1953 in / 1017 out]

✓ Valid JSON — Unified claim record extracted

  CLAIM: EM-CLM-2026-02891
  INSURED: Patricia Yamamoto
  OTHER PARTY: Marcus Webb
  LIABILITY: 0% ours / 100% theirs
  TOTAL RESERVE: $23,500.00
  SUBROGATION: Yes → Pacific Shield Insurance (Marcus Webb policy PS-2024-10283)
  CONFIDENCE: HIGH

  NO DISCREPANCIES — sources are consistent

  DIAGNOSES:
    • Cervical strain/sprain (whiplash) (S13.4XXA)
    • Tension headache secondary to trauma (G44.209)


In [2]:
# =============================================================================
# Template-Based Output Formatting
# =============================================================================
# Instead of letting the model decide the format, we define exact templates
# and have the model fill them in. Like giving a new adjuster a form to
# complete rather than a blank sheet of paper.
# =============================================================================

# --- Template 1: ACORD-style first notice of loss ---
print("=" * 70)
print("TEMPLATE 1: First Notice of Loss (ACORD-Style)")
print("=" * 70)

fnol_template_prompt = f"""Fill in this First Notice of Loss template using 
information from the claim documents below. Use EXACTLY this format — do not 
add or remove any fields. Write "N/A" for any field not found in the documents.

=== FIRST NOTICE OF LOSS ===
Date of Report: [date report was filed]
Claim Number: [claim number]

INSURED INFORMATION
Name: [full name]
Date of Birth: [MM/DD/YYYY]
Policy Number: [policy number]
Carrier: [insurance company]

INCIDENT DETAILS  
Date of Loss: [MM/DD/YYYY]
Time of Loss: [HH:MM AM/PM]
Location: [full address]
Weather Conditions: [conditions at time of incident]
Police Report: [Yes/No] — Report #: [number]

VEHICLE INFORMATION
Year/Make/Model: [insured's vehicle]
License Plate: [plate number]
Drivable: [Yes/No]
Airbag Deployment: [Yes/No]

OTHER PARTIES
Name: [other driver name]
Vehicle: [year make model]
Plate: [plate number]
Insurance: [carrier and policy]
Citations Issued: [Yes/No] — Details: [citation details]

INJURY DETAILS
Insured Injured: [Yes/No]
Description: [injury description]
Treatment Sought: [Yes/No] — Where: [facility name]
Diagnosis: [primary diagnosis]

DAMAGE DESCRIPTION
Insured Vehicle: [description of damage]
Other Vehicle: [description if known]
Third Party Property: [description if applicable]

PRELIMINARY FINANCIALS
Vehicle Repair Estimate: [dollar range]
Medical Costs to Date: [dollar amount]
Initial Reserve: [total reserve amount]

ADJUSTER ASSIGNMENT
Adjuster: [name and license]
Date Assigned: [date]
=== END NOTICE ===

DOCUMENTS:
{police_report}

{medical_record}

{adjuster_notes}"""

result = ask(fnol_template_prompt, max_tokens=1500)
print(result['text'])
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# --- Template 2: Subrogation demand letter ---
print("\n" + "=" * 70)
print("TEMPLATE 2: Subrogation Demand Letter")
print("=" * 70)

demand_prompt = f"""Using the claim information below, fill in this subrogation 
demand letter template. Maintain the formal legal tone. Use EXACTLY this format.

=== SUBROGATION DEMAND ===

[Current Date]

Pacific Shield Insurance
Claims Department
RE: Our Claim #: [claim number]
    Your Insured: [other party name]
    Your Policy #: [other party policy]
    Date of Loss: [date]
    
Dear Claims Representative:

This letter serves as formal demand for reimbursement on behalf of our insured, 
[insured name], for damages arising from the above-referenced incident.

FACTS OF LOSS:
[2-3 sentence summary of the accident, citing the police report number and 
officer's findings]

LIABILITY BASIS:
[2-3 sentences establishing clear liability, referencing the citation issued 
and any supporting evidence]

DAMAGES CLAIMED:
1. Vehicle Repairs (Insured): $[amount or range]
2. Medical Expenses: $[amount to date]
3. Third-Party Property Damage: $[amount or range]
4. Projected Medical (Physical Therapy): $[projected amount]
   Total Demand: $[total]

DOCUMENTATION ENCLOSED:
- Copy of Police Report #[number]
- Vehicle repair estimate  
- Medical records and billing
- Photographs of damage

Please respond within thirty (30) days of receipt. Failure to respond will 
result in escalation including potential arbitration proceedings.

Sincerely,

[Adjuster name]
[Adjuster license number]
Evergreen Mutual Insurance — Subrogation Unit

=== END DEMAND ===

CLAIM DOCUMENTS:
{police_report}

{medical_record}

{adjuster_notes}"""

result = ask(demand_prompt, max_tokens=1500)
print(result['text'])
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# --- Template 3: Policyholder status update email ---
print("\n" + "=" * 70)
print("TEMPLATE 3: Policyholder Status Update Email")
print("=" * 70)

email_prompt = f"""Using the claim information below, fill in this policyholder 
status update email. Keep the tone professional but warm and reassuring.
Use EXACTLY this format.

=== EMAIL ===
TO: [insured name]
FROM: [adjuster name], Evergreen Mutual Claims
SUBJECT: Claim Update — [claim number]

Dear [insured first name],

Thank you for choosing Evergreen Mutual. I wanted to provide you with an 
update on your claim.

CLAIM STATUS: [Active — Under Investigation / Active — Liability Determined / 
Pending / Closed]

WHAT WE'VE DONE:
- [Action 1 — e.g., reviewed police report]
- [Action 2 — e.g., scheduled vehicle inspection]  
- [Action 3 — e.g., assessed liability]

LIABILITY DETERMINATION:
[1-2 sentences summarizing the liability finding in plain, non-legal language]

NEXT STEPS:
1. [Next step 1 with expected timeline]
2. [Next step 2 with expected timeline]
3. [Next step 3 with expected timeline]

WHAT WE NEED FROM YOU:
- [Any items needed from the policyholder, or "No action needed at this time"]

If you have questions, please don't hesitate to reach out at [phone] or reply 
to this email. I'm here to help make this process as smooth as possible.

Warm regards,
[Adjuster name]
Claims Specialist — Evergreen Mutual
=== END EMAIL ===

CLAIM DOCUMENTS:
{police_report}

{medical_record}

{adjuster_notes}"""

result = ask(email_prompt, max_tokens=1500)
print(result['text'])
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

print("\n" + "=" * 70)
print("TEMPLATE TAKEAWAYS")
print("=" * 70)
print("""
Three templates, three audiences, same source data:

1. FNOL FORM    → Internal systems (structured intake)
2. DEMAND LETTER → Other carrier (formal/legal tone)
3. STATUS EMAIL  → Policyholder (warm/reassuring tone)

Why templates beat free-form output:
  • Consistent format across every claim, every adjuster
  • No fields accidentally omitted
  • Easy to validate — check for "N/A" to find missing data
  • Downstream systems can parse known formats reliably
  • New adjusters produce the same quality output as veterans
""")

TEMPLATE 1: First Notice of Loss (ACORD-Style)
=== FIRST NOTICE OF LOSS ===
Date of Report: 02/22/2026
Claim Number: EM-CLM-2026-02891

INSURED INFORMATION
Name: Patricia Yamamoto
Date of Birth: 06/15/1978
Policy Number: EM-AUTO-2025-44891
Carrier: Evergreen Mutual

INCIDENT DETAILS  
Date of Loss: 02/22/2026
Time of Loss: 2:35 PM
Location: NW 23rd Ave & NW Lovejoy St, Portland OR 97210
Weather Conditions: Overcast, dry pavement
Police Report: Yes — Report #: PPB-2026-008834

VEHICLE INFORMATION
Year/Make/Model: 2024 Toyota Camry
License Plate: ABC-1234
Drivable: Yes
Airbag Deployment: No

OTHER PARTIES
Name: Marcus Webb
Vehicle: 2021 Chevrolet Silverado
Plate: XYZ-9876
Insurance: Pacific Shield Insurance, Policy PS-2024-10283
Citations Issued: Yes — Details: Failure to Yield When Entering Roadway (ORS 811.550)

INJURY DETAILS
Insured Injured: Yes
Description: Neck pain and headache
Treatment Sought: Yes — Where: Providence Portland Medical Center
Diagnosis: Cervical strain/sprain (whi

In [3]:
# =============================================================================
# Messy Input Handling & Normalization
# =============================================================================
# Real-world claims don't arrive as clean paragraphs. You get voicemails
# transcribed by speech-to-text, handwritten notes photographed and OCR'd,
# emails with typos, and phone call logs with abbreviations. The model needs
# to normalize all of it into the same clean structure.
# =============================================================================

# --- Messy Input 1: Speech-to-text voicemail transcription ---
print("=" * 70)
print("INPUT 1: Voicemail Transcription (Speech-to-Text Artifacts)")
print("=" * 70)

voicemail = """
yeah hi this is uh mike deluca D E L U C A calling about my my car 
um so i was driving on route nine yesterday no wait it was tuesday 
yeah tuesday afternoon and this deer just came out of nowhere and i 
swerved and hit a guard rail pretty bad um the whole front end is 
messed up the hood is like crumpled and the radiators leaking i had 
it towed to uh pats auto body on on elm street i think my policy 
number is like H B dash seven seven three something i dont have it 
in front of me oh and i also hurt my wrist when the airbag went off 
its pretty swollen but i havent gone to the doctor yet um can someone 
call me back at five oh eight five five five oh one six two thanks bye
"""

print("RAW VOICEMAIL:")
print(voicemail)

vm_prompt = f"""Extract a structured first notice of loss from this voicemail 
transcription. The audio was converted by speech-to-text and contains:
- Filler words (um, uh, like)
- Self-corrections ("yesterday no wait it was tuesday")
- Incomplete information (partial policy number)
- Run-on sentences with no punctuation

Normalize all information and return ONLY valid JSON, no markdown fencing:

{{
  "caller": {{
    "name": "string (properly capitalized)",
    "phone": "string (formatted as XXX-XXX-XXXX)",
    "policy_number": "string or null (note if partial)"
  }},
  "incident": {{
    "date": "string (best estimate, note uncertainty)",
    "location": "string",
    "description": "string (clean, 2-3 sentences)",
    "cause": "string (standardized peril category)"
  }},
  "vehicle": {{
    "damage_description": "string",
    "drivable": boolean,
    "current_location": "string"
  }},
  "injuries": {{
    "reported": boolean,
    "description": "string",
    "treatment_sought": boolean
  }},
  "action_required": ["string"],
  "data_quality": {{
    "completeness": number,
    "issues": ["string"]
  }}
}}

VOICEMAIL TRANSCRIPTION:
{voicemail}"""

result = ask(vm_prompt)
try:
    parsed = parse_llm_json(result['text'])
    print("\nEXTRACTED (CLEAN):")
    print(json.dumps(parsed, indent=2))
    print(f"\n  Completeness: {parsed['data_quality']['completeness']}")
    print(f"  Issues: {len(parsed['data_quality']['issues'])}")
    for issue in parsed['data_quality']['issues']:
        print(f"    ⚠ {issue}")
except json.JSONDecodeError as e:
    print(f"✗ JSON parse error: {e}")
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# --- Messy Input 2: OCR'd handwritten adjuster field notes ---
print("\n" + "=" * 70)
print("INPUT 2: OCR'd Handwritten Field Notes (Recognition Errors)")
print("=" * 70)

ocr_notes = """
FIEL0 INSPECTI0N N0TES
Adjstr: R. Tbrres  Dt: 02/2B/2026

Prop: 1847 0ak1and Ave, Unit 3
Ins'd: Jennifor Paik  Po1#: HO-2O25-991O3

OBSERVATI0NS:
- R00f: Missing shing1es on S/W corner approx 15 sq ft
  Exposed under1ayment visib1e, no active 1eak YET
- Gutters: Pu11ed away from fascia on south side ~20 1in ft  
- Siding: Impact damage E side, 3 pane1s cracked
  Loks like hai1 damage — circuIar pattern
- Interior: No water damage observed at time of insp
  Homeownr says she heard "banging" during storm 02/25
- Windw: 1 broken pane — master BR, east facing
  Temp repair w/ p1astic sheeting a1ready done by ins'd

PRELI M EST: $8,5OO - $11,OOO
Recommend: Fu11 roof inspection by certified roofer
Ph0tos: 47 taken, up1oaded to c1aim fi1e

PRIORITY: MEDIUN — no active water intrusion but 
exposed r00f needs tarp before next rain (forecast Wed)
"""

print("RAW OCR OUTPUT:")
print(ocr_notes)

ocr_prompt = f"""Extract structured data from these field inspection notes. 
The text was produced by OCR from handwritten notes and contains:
- Character substitution errors (0 for O, 1 for l, B for 8, etc.)
- Abbreviated words (Adjstr, Prop, Ins'd, etc.)
- Misspellings from OCR misreads
- Informal shorthand

Correct all OCR errors and normalize into clean structured data.
Return ONLY valid JSON, no markdown fencing:

{{
  "inspection": {{
    "adjuster": "string (corrected name)",
    "date": "YYYY-MM-DD (corrected)",
    "property_address": "string (corrected)"
  }},
  "insured": {{
    "name": "string (corrected spelling)",
    "policy_number": "string (corrected)"
  }},
  "findings": [
    {{
      "area": "string",
      "damage_type": "string",
      "description": "string (clean, corrected)",
      "severity": "string (minor/moderate/severe)",
      "measurements": "string or null"
    }}
  ],
  "interior_damage": boolean,
  "cause_of_loss": "string (standardized)",
  "preliminary_estimate": {{"low": number, "high": number}},
  "recommendations": ["string"],
  "priority": "string (LOW/MEDIUM/HIGH/URGENT)",
  "urgency_reason": "string or null",
  "photos_taken": number,
  "ocr_corrections_made": ["string"]
}}

OCR TEXT:
{ocr_notes}"""

result = ask(ocr_prompt, max_tokens=2000)
try:
    parsed = parse_llm_json(result['text'])
    print("\nEXTRACTED (CLEAN):")
    print(json.dumps(parsed, indent=2))
    print(f"\n  Findings: {len(parsed['findings'])} areas inspected")
    print(f"  Estimate: ${parsed['preliminary_estimate']['low']:,} — ${parsed['preliminary_estimate']['high']:,}")
    print(f"  Priority: {parsed['priority']}")
    if parsed.get('ocr_corrections_made'):
        print(f"\n  OCR CORRECTIONS ({len(parsed['ocr_corrections_made'])}):")
        for c in parsed['ocr_corrections_made']:
            print(f"    → {c}")
except json.JSONDecodeError as e:
    print(f"✗ JSON parse error: {e}")
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# --- Messy Input 3: Abbreviated email chain ---
print("\n" + "=" * 70)
print("INPUT 3: Internal Email Chain (Abbreviations & Jargon)")
print("=" * 70)

email_chain = """
From: t.nguyen@evergreenins.com
To: r.torres@evergreenins.com
Date: 02/27/26 4:47 PM
Subject: RE: RE: Yamamoto clm - heads up

Rach — 

fyi got a call from Pac Shield adjuster re: the Yamamoto/Webb thing. 
They're pushing back on 100% liab on their guy. Saying our ins'd was 
going 32 in a 25 which "contributed." lol ok. I sent them the dashcam 
+ traffic cam. Ball's in their court.

Also Webb's atty (some firm called Blackwell & Assoc) sent a rep letter 
today. So he's lawyered up. Doesn't change our liab position but heads 
up for your file.

Separately — the Chow subrg demand from Cascade came in at $2,650. 
Seems reasonable, w/in our reserve. Recommend we just pay it and close 
that piece out.

Oh and Yamamoto called asking abt rental. She's in the collar still, 
doing PT 2x/wk. Said the shop (Eastside Auto) told her parts are 
backordered, looking at 3-4 wks. I told her we'd authorize Enterprise 
mid-size, 30 days max. Pls confirm.

-Tony

---

From: r.torres@evergreenins.com  
To: t.nguyen@evergreenins.com
Date: 02/27/26 5:12 PM
Subject: RE: RE: RE: Yamamoto clm - heads up

Tony — thx for the update. Agree on all fronts:

1. Liab stands at 100% Webb. 2mph over is w/in margin, won't move the 
   needle in arb. Good call sending the footage.
2. Note the rep letter in file. Flag for litigation if they file.
3. Approve Cascade subrg $2,650. Process payment.
4. Rental auth'd — Enterprise mid-size, 30 day max. Log it.

One more thing — can you bump the BI reserve to $15K? PT is 2x/wk 
and she's still in a collar at 5 wks post-accident. This might run 
longer than we initially thought.

-RT
"""

print("RAW EMAIL CHAIN:")
print(email_chain)

email_prompt = f"""Extract all actionable information from this internal email 
chain. The emails contain:
- Insurance abbreviations (clm, liab, subrg, ins'd, BI, arb, PT, rep letter)
- Informal tone and shorthand
- Multiple distinct action items mixed into conversational text
- References to an existing claim file

Decode all abbreviations and extract structured data.
Return ONLY valid JSON, no markdown fencing:

{{
  "claim_reference": "string",
  "participants": [
    {{"name": "string", "email": "string", "role": "string"}}
  ],
  "key_updates": [
    {{
      "topic": "string",
      "summary": "string (decoded from abbreviations, plain language)",
      "source_email": number
    }}
  ],
  "action_items": [
    {{
      "action": "string (clear, specific)",
      "assigned_to": "string",
      "status": "string (approved/pending/needs_review)",
      "deadline": "string or null"
    }}
  ],
  "financial_changes": [
    {{
      "type": "string",
      "old_value": "number or null",
      "new_value": number,
      "reason": "string"
    }}
  ],
  "litigation_flags": {{
    "attorney_involved": boolean,
    "firm_name": "string or null",
    "representing": "string",
    "action_needed": "string"
  }},
  "abbreviations_decoded": {{}}
}}

EMAIL CHAIN:
{email_chain}"""

result = ask(email_prompt, max_tokens=2000)
try:
    parsed = parse_llm_json(result['text'])
    print("\nEXTRACTED (CLEAN):")
    print(json.dumps(parsed, indent=2))
    print(f"\n  Updates: {len(parsed['key_updates'])}")
    print(f"  Action Items: {len(parsed['action_items'])}")
    print(f"  Financial Changes: {len(parsed['financial_changes'])}")
    print(f"  Attorney Involved: {parsed['litigation_flags']['attorney_involved']}")
    if parsed.get('abbreviations_decoded'):
        print(f"\n  ABBREVIATIONS DECODED ({len(parsed['abbreviations_decoded'])}):")
        for abbr, meaning in parsed['abbreviations_decoded'].items():
            print(f"    {abbr} → {meaning}")
except json.JSONDecodeError as e:
    print(f"✗ JSON parse error: {e}")
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

print("\n" + "=" * 70)
print("MESSY INPUT TAKEAWAYS")
print("=" * 70)
print("""
Three types of real-world noise, one clean output:

1. VOICEMAIL (speech-to-text)  → Filler words, self-corrections, no punctuation
2. OCR FIELD NOTES (handwriting) → Character substitutions, abbreviations, misspellings
3. EMAIL CHAIN (human shorthand) → Jargon, abbreviations, conversational tone

Key prompt techniques for messy input:
  • TELL the model what kind of noise to expect (it handles it better)
  • ASK for a data quality or corrections section (transparency)
  • USE the same output schema regardless of input quality
  • INCLUDE action_required / action_items to drive workflows

In production, the input channel tags the noise type automatically:
  voicemail → "speech-to-text artifacts"
  scanned doc → "OCR recognition errors"  
  email → "informal abbreviations"
""")

INPUT 1: Voicemail Transcription (Speech-to-Text Artifacts)
RAW VOICEMAIL:

yeah hi this is uh mike deluca D E L U C A calling about my my car 
um so i was driving on route nine yesterday no wait it was tuesday 
yeah tuesday afternoon and this deer just came out of nowhere and i 
swerved and hit a guard rail pretty bad um the whole front end is 
messed up the hood is like crumpled and the radiators leaking i had 
it towed to uh pats auto body on on elm street i think my policy 
number is like H B dash seven seven three something i dont have it 
in front of me oh and i also hurt my wrist when the airbag went off 
its pretty swollen but i havent gone to the doctor yet um can someone 
call me back at five oh eight five five five oh one six two thanks bye


EXTRACTED (CLEAN):
{
  "caller": {
    "name": "Mike DeLuca",
    "phone": "508-555-0162",
    "policy_number": "HB-773*** (partial)"
  },
  "incident": {
    "date": "Tuesday (exact date unknown - caller stated 'yesterday' then correct

In [4]:
# =============================================================================
# Format Validation Patterns
# =============================================================================
# Getting JSON back from the model is step one. Validating that the JSON
# contains what you expect — correct types, required fields, valid ranges —
# is step two. This is the bridge between "it looks right" and "it IS right."
# =============================================================================

# --- A reusable validation framework ---

def validate_claim_json(data, schema_rules):
    """
    Validate extracted claim data against business rules.
    Returns a list of validation errors (empty = valid).
    
    Think of this like form validation on a website — the form might 
    submit, but did the user put a valid email in the email field?
    """
    errors = []
    warnings = []
    
    for rule in schema_rules:
        field_path = rule['field']
        
        # Navigate nested keys (e.g., "claimant.name" → data['claimant']['name'])
        value = data
        try:
            for key in field_path.split('.'):
                if isinstance(value, dict):
                    value = value[key]
                else:
                    raise KeyError(key)
        except (KeyError, TypeError):
            if rule.get('required', False):
                errors.append(f"MISSING REQUIRED: {field_path}")
            continue
        
        # Type check
        if 'type' in rule and value is not None:
            if not isinstance(value, rule['type']):
                errors.append(f"WRONG TYPE: {field_path} — expected {rule['type'].__name__}, got {type(value).__name__}")
                continue
        
        # Null check for required fields
        if rule.get('required', False) and value is None:
            errors.append(f"NULL REQUIRED: {field_path} — value cannot be null")
            continue
        
        # Range check for numbers
        if 'min' in rule and value is not None and isinstance(value, (int, float)):
            if value < rule['min']:
                errors.append(f"OUT OF RANGE: {field_path} = {value} (min: {rule['min']})")
        if 'max' in rule and value is not None and isinstance(value, (int, float)):
            if value > rule['max']:
                errors.append(f"OUT OF RANGE: {field_path} = {value} (max: {rule['max']})")
        
        # Allowed values check
        if 'allowed' in rule and value is not None:
            if value not in rule['allowed']:
                warnings.append(f"UNEXPECTED VALUE: {field_path} = '{value}' (expected one of: {rule['allowed']})")
        
        # Pattern check for strings
        if 'pattern' in rule and value is not None and isinstance(value, str):
            if not re.match(rule['pattern'], value):
                errors.append(f"BAD FORMAT: {field_path} = '{value}' (expected pattern: {rule['pattern']})")
    
    return errors, warnings

# --- Define validation rules for a claim extraction ---
claim_rules = [
    # Required fields
    {'field': 'claim.claim_number', 'required': True, 'type': str, 
     'pattern': r'^[A-Z]{2}-CLM-\d{4}-\d{5}$'},
    {'field': 'claim.incident_date', 'required': True, 'type': str,
     'pattern': r'^\d{4}-\d{2}-\d{2}$'},
    {'field': 'insured.name', 'required': True, 'type': str},
    {'field': 'insured.policy_number', 'required': True, 'type': str},
    
    # Liability must be valid
    {'field': 'liability.our_insured_pct', 'required': True, 'type': (int, float),
     'min': 0, 'max': 100},
    {'field': 'liability.other_party_pct', 'required': True, 'type': (int, float),
     'min': 0, 'max': 100},
    
    # Financial validation
    {'field': 'financials.reserves.total', 'required': True, 'type': (int, float),
     'min': 0, 'max': 10000000},
    {'field': 'financials.medical_costs_to_date', 'required': True, 'type': (int, float),
     'min': 0},
    
    # Confidence must be a known value
    {'field': 'extraction_confidence', 'required': True, 'type': str,
     'allowed': ['HIGH', 'MEDIUM', 'LOW']},
]

# --- Test 1: Validate the good extraction from Cell 1 ---
print("=" * 70)
print("TEST 1: Validate a Clean Extraction")
print("=" * 70)

# Re-run the Cell 1 extraction 
extraction_prompt = f"""You have three documents about the same auto insurance claim.
Cross-reference them and extract a unified claim record as JSON.
Return ONLY valid JSON, no markdown fencing.

Schema:
{{
  "claim": {{
    "claim_number": "string",
    "incident_date": "YYYY-MM-DD",
    "incident_time": "HH:MM",
    "location": "string",
    "police_report_number": "string"
  }},
  "insured": {{
    "name": "string",
    "policy_number": "string"
  }},
  "other_party": {{
    "name": "string",
    "policy_number": "string"
  }},
  "liability": {{
    "our_insured_pct": number,
    "other_party_pct": number,
    "basis": "string"
  }},
  "financials": {{
    "reserves": {{"property_damage": number, "medical_bi": number, "total": number}},
    "medical_costs_to_date": number,
    "vehicle_repair_estimate": {{"low": number, "high": number}}
  }},
  "extraction_confidence": "HIGH | MEDIUM | LOW"
}}

DOCUMENT 1 — POLICE REPORT:
{police_report}

DOCUMENT 2 — MEDICAL RECORD:
{medical_record}

DOCUMENT 3 — ADJUSTER NOTES:
{adjuster_notes}"""

result = ask(extraction_prompt, max_tokens=2000)
try:
    clean_data = parse_llm_json(result['text'])
    errors, warnings = validate_claim_json(clean_data, claim_rules)
    
    print(f"Extraction: ✓ Valid JSON")
    print(f"Validation: {len(errors)} errors, {len(warnings)} warnings")
    
    if errors:
        print("\nERRORS:")
        for e in errors:
            print(f"  ✗ {e}")
    if warnings:
        print("\nWARNINGS:")
        for w in warnings:
            print(f"  ⚠ {w}")
    if not errors and not warnings:
        print("  ✓ All validation rules passed")
        
except json.JSONDecodeError as e:
    print(f"✗ JSON parse error: {e}")

print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# --- Test 2: Validate intentionally bad data ---
print("\n" + "=" * 70)
print("TEST 2: Validate Intentionally Bad Data")
print("=" * 70)

bad_data = {
    "claim": {
        "claim_number": "BAD-FORMAT-123",
        "incident_date": "February 22, 2026"
    },
    "insured": {
        "name": None,
        "policy_number": "EM-AUTO-2025-44891"
    },
    "liability": {
        "our_insured_pct": 150,
        "other_party_pct": -50
    },
    "financials": {
        "reserves": {"total": -5000},
        "medical_costs_to_date": 2840
    },
    "extraction_confidence": "MAYBE"
}

print("Input data (intentionally broken):")
print(json.dumps(bad_data, indent=2))

errors, warnings = validate_claim_json(bad_data, claim_rules)

print(f"\nValidation: {len(errors)} errors, {len(warnings)} warnings")
print("\nERRORS:")
for e in errors:
    print(f"  ✗ {e}")
print("\nWARNINGS:")
for w in warnings:
    print(f"  ⚠ {w}")

# --- Test 3: Retry pattern — when validation fails, ask the model to fix it ---
print("\n" + "=" * 70)
print("TEST 3: Auto-Retry on Validation Failure")  
print("=" * 70)

def extract_with_validation(prompt, rules, max_retries=2):
    """
    Extract JSON from LLM, validate it, and retry with error feedback 
    if validation fails. Like sending a form back to the employee with 
    red circles around the mistakes.
    """
    for attempt in range(max_retries + 1):
        if attempt == 0:
            result = ask(prompt, max_tokens=2000)
        else:
            # Feed errors back to the model and ask it to fix them
            retry_prompt = f"""Your previous JSON extraction had these validation errors:

{chr(10).join(f'- {e}' for e in errors)}

Here was your output:
{result['text']}

Please fix these errors and return corrected JSON only, no markdown fencing.
Remember:
- claim_number format must be: XX-CLM-YYYY-NNNNN
- incident_date must be: YYYY-MM-DD
- liability percentages must be 0-100
- all financial values must be >= 0
- extraction_confidence must be HIGH, MEDIUM, or LOW"""
            
            result = ask(retry_prompt, max_tokens=2000)
        
        try:
            data = parse_llm_json(result['text'])
            errors, warnings = validate_claim_json(data, rules)
            
            status = "PASS" if not errors else "FAIL"
            print(f"\n  Attempt {attempt + 1}: {status} — {len(errors)} errors, {len(warnings)} warnings")
            
            if not errors:
                return data, attempt + 1, warnings
                
        except json.JSONDecodeError:
            errors = ["JSON parse failed"]
            print(f"\n  Attempt {attempt + 1}: FAIL — invalid JSON")
    
    return None, max_retries + 1, []

# Run the extraction with validation and auto-retry
print("Running extraction with auto-retry...")
data, attempts, warnings = extract_with_validation(extraction_prompt, claim_rules)

if data:
    print(f"\n✓ Valid extraction in {attempts} attempt(s)")
    if warnings:
        for w in warnings:
            print(f"  ⚠ {w}")
else:
    print(f"\n✗ Failed after {attempts} attempts — needs human review")

# --- Summary ---
print("\n" + "=" * 70)
print("VALIDATION PATTERN SUMMARY")
print("=" * 70)
print("""
Three layers of defense:

1. JSON PARSING (parse_llm_json)
   → Can we even read the output? Strip fencing, parse JSON.

2. SCHEMA VALIDATION (validate_claim_json)
   → Is the data correct? Required fields, types, ranges, formats.

3. AUTO-RETRY (extract_with_validation)
   → If validation fails, feed errors back and let the model fix itself.
   → Like sending a form back with red circles on the mistakes.

Production considerations:
  • Set max_retries to 2-3 (diminishing returns after that)
  • Log every retry for monitoring (are certain fields failing often?)
  • If retries exhaust, route to human review queue
  • Track validation pass rate as a system health metric
  • Consider Pydantic for complex validation (remember Day 16 review)
""")

TEST 1: Validate a Clean Extraction
Extraction: ✓ Valid JSON
Validation: 0 errors, 0 warnings
  ✓ All validation rules passed

[Tokens: 1557 in / 389 out]

TEST 2: Validate Intentionally Bad Data
Input data (intentionally broken):
{
  "claim": {
    "claim_number": "BAD-FORMAT-123",
    "incident_date": "February 22, 2026"
  },
  "insured": {
    "name": null,
    "policy_number": "EM-AUTO-2025-44891"
  },
  "liability": {
    "our_insured_pct": 150,
    "other_party_pct": -50
  },
  "financials": {
    "reserves": {
      "total": -5000
    },
    "medical_costs_to_date": 2840
  },
  "extraction_confidence": "MAYBE"
}

Validation: 6 errors, 1 warnings

ERRORS:
  ✗ BAD FORMAT: claim.claim_number = 'BAD-FORMAT-123' (expected pattern: ^[A-Z]{2}-CLM-\d{4}-\d{5}$)
  ✗ BAD FORMAT: claim.incident_date = 'February 22, 2026' (expected pattern: ^\d{4}-\d{2}-\d{2}$)
  ✗ NULL REQUIRED: insured.name — value cannot be null
  ✗ OUT OF RANGE: liability.our_insured_pct = 150 (max: 100)
  ✗ OUT OF RANG